# 03 · Interpretasi Model & Evaluasi Akhir (Test Set)

Melanjutkan dari `02_modeling.ipynb`. Notebook ini memuat kembali model dan data yang sudah disimpan (tanpa training ulang), lalu melakukan interpretasi (feature importance / SHAP) dan evaluasi akhir yang tidak bias pada test set.

 Setup — muat kembali model & data

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../src')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, confusion_matrix, classification_report
)
import shap

from utils import evaluate_model

best_estimators = joblib.load('../models/all_models.pkl')
comparison_df = pd.read_csv('../models/comparison_val_results.csv', index_col=0)

X_val_fe = pd.read_csv('../data/processed/X_val_fe.csv')
X_test_fe = pd.read_csv('../data/processed/X_test_fe.csv')
y_val = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print('Model & data berhasil dimuat.')
comparison_df.sort_values('ROC-AUC', ascending=False)

## 11. Interpretasi Model (Feature Importance & SHAP)

Interpretasi dilakukan pada model dengan ROC-AUC tertinggi di validation set untuk melihat
fitur apa yang paling berpengaruh terhadap prediksi risiko kredit.

In [ ]:
best_model_name = comparison_df['ROC-AUC'].idxmax()
best_model = best_estimators[best_model_name]
print('Model dengan ROC-AUC tertinggi (validation):', best_model_name)

In [ ]:
# Ambil nama fitur hasil preprocessing (setelah one-hot encoding)
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
classifier = best_model.named_steps['classifier']

if hasattr(classifier, 'feature_importances_'):
    importances = pd.Series(classifier.feature_importances_, index=feature_names)
    top_features = importances.sort_values(ascending=False).head(15)

    plt.figure(figsize=(8,6))
    sns.barplot(x=top_features.values, y=top_features.index, color='steelblue')
    plt.title(f'Top 15 Feature Importance - {best_model_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    coefs = pd.Series(classifier.coef_[0], index=feature_names)
    top_features = coefs.reindex(coefs.abs().sort_values(ascending=False).head(15).index)

    plt.figure(figsize=(8,6))
    sns.barplot(x=top_features.values, y=top_features.index, color='steelblue')
    plt.title(f'Top 15 Feature Coefficients - {best_model_name}')
    plt.xlabel('Coefficient (log-odds)')
    plt.tight_layout()
    plt.show()

In [ ]:
# SHAP explanation untuk model terbaik
X_val_transformed = best_model.named_steps['preprocessor'].transform(X_val_fe)
X_val_transformed = pd.DataFrame(X_val_transformed, columns=feature_names)

try:
    explainer = shap.TreeExplainer(classifier)
    shap_values = explainer.shap_values(X_val_transformed)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    shap.summary_plot(shap_values, X_val_transformed, show=False, max_display=15)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print('SHAP TreeExplainer tidak berlaku untuk model ini (mis. Logistic Regression):', e)
    explainer = shap.LinearExplainer(classifier, X_val_transformed)
    shap_values = explainer.shap_values(X_val_transformed)
    shap.summary_plot(shap_values, X_val_transformed, show=False, max_display=15)
    plt.tight_layout()
    plt.show()

## 12. Pemilihan Model Terbaik & Evaluasi Akhir di Test Set

**Kriteria pemilihan:** untuk kasus credit risk, kesalahan yang paling mahal adalah
*False Negative* (menyetujui kredit nasabah yang sebenarnya berisiko *bad*). Oleh karena
itu model terbaik dipilih berdasarkan kombinasi **ROC-AUC** (kemampuan membedakan kelas
secara keseluruhan) dan **Recall kelas Bad** (kemampuan menangkap nasabah berisiko),
bukan semata-mata Accuracy yang bisa menyesatkan pada data imbalanced.

Test set baru dibuka sekali di tahap ini, sebagai estimasi performa akhir yang tidak bias
oleh proses tuning maupun pemilihan model.

In [ ]:
print('=== Ringkasan Perbandingan Model (Validation Set) ===')
display(comparison_df.sort_values('ROC-AUC', ascending=False))

print(f'\n>>> Model terpilih: {best_model_name} <<<')
print('Alasan: ROC-AUC dan/atau Recall (Bad) tertinggi pada validation set di antara ketiga model.')

In [ ]:
# Evaluasi akhir di test set (baru dibuka sekali)
test_result = evaluate_model(best_model, X_test_fe, y_test, best_model_name)
print('=== Evaluasi Akhir di Test Set ===')
for k, v in test_result.items():
    if k != 'Model':
        print(f'{k}: {v:.4f}')

print(f'\nClassification Report - {best_model_name} (Test Set):')
print(classification_report(y_test, best_model.predict(X_test_fe), target_names=['Good(0)','Bad(1)']))

cm_test = confusion_matrix(y_test, best_model.predict(X_test_fe))
plt.figure(figsize=(5,4))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Good(0)','Bad(1)'], yticklabels=['Good(0)','Bad(1)'])
plt.title(f'Confusion Matrix - {best_model_name} (Test Set)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Kesimpulan

- Ketiga model (Logistic Regression, Random Forest, XGBoost) berhasil dilatih dengan
  hyperparameter tuning berbasis cross-validation dan dievaluasi secara adil menggunakan
  validation set yang terpisah dari proses tuning.
- Model terbaik dipilih menggunakan ROC-AUC & Recall kelas Bad sebagai prioritas metrik,
  sesuai konteks bisnis credit risk, kemudian diuji sekali di test set untuk estimasi
  performa akhir yang tidak bias.
- Interpretasi model (feature importance / SHAP) menunjukkan status rekening giro/tabungan,
  jumlah kredit, durasi, dan fitur turunan `Credit_per_Month` sebagai pendorong risiko
  utama — konsisten dengan insight EDA di Bagian 4.
- Untuk pengembangan lebih lanjut: dapat dicoba teknik penyeimbangan kelas seperti SMOTE,
  atau ensembel gabungan (stacking) antar model.